# 6. Hyperparameter Tuning

ทำ tuning จาก **17 raw input features** ด้วย `Pipeline` ทั้งก้อน เพื่อให้ preprocessing ถูก fit ใหม่ภายในแต่ละ cross-validation fold และไม่เห็น validation fold ล่วงหน้า

In [11]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline

train_df = pd.read_parquet("../data/processed/train_pre_impute.parquet")
valid_df = pd.read_parquet("../data/processed/valid_pre_impute.parquet")
test_df = pd.read_parquet("../data/processed/test_pre_impute.parquet")

X_train = train_df.drop(columns="Dropout_Risk")
y_train = train_df["Dropout_Risk"]
X_valid = valid_df.drop(columns="Dropout_Risk")
y_valid = valid_df["Dropout_Risk"]
X_test = test_df.drop(columns="Dropout_Risk")
y_test = test_df["Dropout_Risk"]

preprocessor_path = "../models/preprocessing/preprocessor.joblib"


In [12]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

dt_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", DecisionTreeClassifier(random_state=42)),
])

dt_param_dist = {
    "model__max_depth": [3, 5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5, 10],
}

dt_search = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=dt_param_dist,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
)
dt_search.fit(X_train, y_train)
print("DT best params:", dt_search.best_params_)
print("DT best CV accuracy:", dt_search.best_score_)


DT best params: {'model__min_samples_split': 2, 'model__min_samples_leaf': 10, 'model__max_depth': 10}
DT best CV accuracy: 0.6321428571428571


In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", RandomForestClassifier(random_state=42)),
])

rf_param_dist = {
    "model__n_estimators": [50, 100, 200, 300],
    "model__max_depth": [3, 5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", None],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="accuracy",
    random_state=42,
)
rf_search.fit(X_train, y_train)
print("RF best params:", rf_search.best_params_)
print("RF best CV accuracy:", rf_search.best_score_)


RF best params: {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 5}
RF best CV accuracy: 0.7107142857142857


In [14]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("preprocessor", joblib.load(preprocessor_path)),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

lr_param_dist = {
    "model__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_dist,
    n_iter=6,
    cv=5,
    scoring="accuracy",
    random_state=42,
)
lr_search.fit(X_train, y_train)
print("LR best params:", lr_search.best_params_)
print("LR best CV accuracy:", lr_search.best_score_)


LR best params: {'model__C': 0.01}
LR best CV accuracy: 0.7125


## Model selection on validation set

Cross-validation ใช้เฉพาะ training set สำหรับ tuning จากนั้นใช้ validation set เพื่อเลือกระหว่าง baseline LR และ tuned candidates ส่วน **test set จะยังไม่ถูกใช้จนกว่าจะเลือก final configuration เสร็จ**

In [15]:
baseline_lr = joblib.load("../models/baseline/lr_pipeline.joblib")

candidates = {
    "Baseline Logistic Regression": baseline_lr,
    "Tuned Logistic Regression": lr_search.best_estimator_,
    "Tuned Decision Tree": dt_search.best_estimator_,
    "Tuned Random Forest": rf_search.best_estimator_,
}

validation_rows = []
for name, estimator in candidates.items():
    validation_rows.append({
        "Model": name,
        "Train Accuracy": estimator.score(X_train, y_train),
        "Validation Accuracy": estimator.score(X_valid, y_valid),
    })

validation_results = pd.DataFrame(validation_rows).sort_values(
    "Validation Accuracy", ascending=False
).reset_index(drop=True)
validation_results

,Model,Train Accuracy,Validation Accuracy
0,Tuned Logistic Regression,0.737500,0.683333
1,Tuned Random Forest,0.878571,0.675000
2,Baseline Logistic Regression,0.733929,0.658333
3,Tuned Decision Tree,0.808929,0.591667


In [16]:
best_name = validation_results.loc[0, "Model"]
selected_pipeline = candidates[best_name]
print("Selected model:", best_name)

Selected model: Tuned Logistic Regression


## Final model and test evaluation

หลังเลือกโมเดลจาก validation set แล้ว จะ fit โมเดลที่เลือกอีกครั้งด้วย **train + validation** จากนั้นประเมินกับ test set เพียงครั้งเดียวเพื่อรายงานผลสุดท้าย


In [17]:
X_train_final = pd.concat([X_train, X_valid], axis=0)
y_train_final = pd.concat([y_train, y_valid], axis=0)

# fit โมเดลที่เลือกใหม่ด้วย train + validation
final_pipeline = selected_pipeline
final_pipeline.fit(X_train_final, y_train_final)

print("Final test accuracy:", final_pipeline.score(X_test, y_test))


Final test accuracy: 0.6333333333333333


In [18]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = final_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.62      0.53      0.57        55
         Yes       0.64      0.72      0.68        65

    accuracy                           0.63       120
   macro avg       0.63      0.63      0.62       120
weighted avg       0.63      0.63      0.63       120

Confusion matrix:
[[29 26]
 [18 47]]


In [19]:
from pathlib import Path

Path("../models/final").mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, "../models/final/dropout_pipeline.joblib")
print("Saved: ../models/final/dropout_pipeline.joblib")

Saved: ../models/final/dropout_pipeline.joblib


In [20]:
# Sanity check: โหลด artifact แล้ว predict raw rows ได้ทันที
loaded_pipeline = joblib.load("../models/final/dropout_pipeline.joblib")
print(loaded_pipeline.predict(X_test.head(5)))

['No' 'No' 'No' 'Yes' 'Yes']
